## LABORATORIO 3 BSAS & MBSAS

Nuria Arroyo Bustamante 

Apredizaje de Máquina  - Otoño 2025

Dr. Hector Saib Gómez Maravillo 


## Objetivo final

**Objetivo final**: Implementar el BSAS y el MBSAS en Python.

## Instrucciones

**Instrucciones**:

1. Utiliza los datos analizados en el laboratorio 2.

2. Elige el tipo de función de disimilitud/similitud entre punto-punto; punto-conjunto y conjunto-conjunto a utilizar.

3. Calcular la matriz de disimilitud/similitud. Esto puede realizarse dentro o fuera de los algoritmos.

4. Implementa los algoritmos BSAS y MBSAS como dos funciones de Python con las siguientes características:

   a. Entrada:

   - Un conjunto de datos o la matriz de disimilitud/similitud  
   - Parámetros correspondientes.  
   - Un booleano para determinar si la exploración de datos se hace en orden ascendente (0,1,2,3,....,n-1) o en orden aleatorio.  
   - Recomendación usar numpy.random.shuffle

   b. La función debe usar la matriz de disimilitud/similitud para implementar el algoritmo BSAS o MBSAS.

   c. Salida:

   - Una representación adecuada de la partición o clustering resultante de aplicar el algoritmo.  
   - Si la matriz de disimilitud/similitud se calcula dentro de la función, entonces también debe devolverse.

5. Aplica un método de reducción de dimensionalidad para graficar los datos originales y el clustering resultante utilizando un color diferente para cada clúster.

6. Crea una función que realice la estimación gráfica del mejor número de clústeres.

   d. Parámetros:

   - Matriz de similitud/disimilitud  
   - Número máximo de clusters q  
   - Tamaño de paso c  
   - Tipo de algoritmo MBSAS/BSAS

   e. Funcionamiento:

   - La función debe calcular la distancia mínima (a) y máxima (b) de la función (puede usarse .min, .max)  
   - Después debe ejecutar el algoritmo seleccionado iterativamente para todos los valores de ﻿theta﻿ desde ﻿a﻿ hasta ﻿b﻿, utilizando ﻿c﻿ como tamaño de paso.  
   - Gráficar el número de clusteres (eje y) en función del valor de ﻿theta﻿ (eje x).  
   - Comparar las gráficas para el MBSAS y el BSAS, para un valor de q específico.


In [ ]:
#libraries 
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import umap.umap_ as umap
from sklearn.manifold import TSNE
import plotly.express as px

Se importa la matriz de similitud del csv del lab 2 y se crea la de disimilitud

In [186]:
#leer la matriz de similitud del csv del lab 2 y crear la de disimilitud
matriz_similitud = pd.read_csv('matriz_similitud_wl3.csv', index_col=False)
matriz_similitud = matriz_similitud.to_numpy()
D = 1 - matriz_similitud

In [187]:
shape = D.shape
print(f"La matriz de disimilitud es de tamaño: {shape}")
# esto para ver que la matriz se leyó bien y es cuadrada
# si no puede causar problemas de indexación

La matriz de disimilitud es de tamaño: (1000, 1000)


Creacion de las funciones auxiliares y de las funciones principales BSAS y MBSAS algoritmos secuanciales de clustering vistos en clase

In [188]:
#funciiones auxiliares 

#distancia entre un punto y un conjunto
def punto_a_conjunto(C, a, D):
    #C: conjunto
    #a: punto
    #D: matriz de disimilitud
    sum = 0
    c = 0
    for x in C:
        sum += D[a][x]
        c += 1
    return sum / c if c > 0 else float('inf')

#funcion para generar un nuevo orden aleatorio de indices
def nuevo_orden(D):
    indices = list(range(len(D)))
    np.random.shuffle(indices)
    return indices


In [189]:
#BSAS
def bsas(D, theta, q, random_order=False):
    """Los parametros de la función son:
    D: matriz de disimilitud
    theta: umbral de disimilitud
    q: número máximo de clusters
    random_order: booleano para determinar si el orden de exploración es aleatorio
    """
    m = 1 # empieza la cuenta de clusters
    if random_order == True: # si es True, se genera un nuevo orden aleatorio con la función nuevo_orden
        indices = nuevo_orden(D)
    else: # si es False, se usa el orden natural que no tiene significado es solo como venian en la base de datos 
        indices = list(range(len(D)))   
    C = []  # lista de clusters 
    C.append({indices[0]})  # primer cluster con el primer punto
    for i in indices[1:]: # recorrer los puntos en el orden dado
        # encontrar el cluster mas cercano entonces se setan los valores iniciales
        min_dist = float('inf')
        closest_cluster = -1
        for j in range(m):
            dist = punto_a_conjunto(C[j], i, D)
            if dist < min_dist:
                min_dist = dist
                closest_cluster = j
                #una vez que se tiene el cluster mas cercano se compara la distancia minima con el umbral
        if min_dist < theta: # si la distancia minima es menor que el umbral
            C[closest_cluster].add(i) # se agrega el punto al cluster mas cercano
        else: # si no
             # se crea un nuevo cluster si no se ha alcanzado el maximo numero de clusters
            if m < q:
                C.append({i})
                m += 1
            else: # si se ha alcanzado el maximo numero de clusters
                 # se agrega el punto al cluster mas cercano aunque la distancia sea mayor que el umbral
                C[closest_cluster].add(i)
    return C


In [190]:
# Ejemplo de uso
bsas_c = bsas(D, 0.4, 20, random_order=True)
print("Número de clusters BSAS:", len(bsas_c))

Número de clusters BSAS: 20


In [191]:
#MBSAS
def mbsas(D, theta, random_order=False):
    """Los parametros de la función son:
    D: matriz de disimilitud
    theta: umbral de disimilitud
    random_order: booleano para determinar si el orden de exploración es aleatorio
    """
    m = 1
    if random_order == True:
        indices = nuevo_orden(D)
    else:
        indices = list(range(len(D)))   
    C = []  # lista de clusters 
    C.append({indices[0]})  # primer cluster con el primer punto
    for i in indices[1:]:
        # encontrar el cluster mas cercano
        min_dist = float('inf')
        closest_cluster = -1
        for j in range(m):
            dist = punto_a_conjunto(C[j], i, D)
            if dist < min_dist:
                min_dist = dist
                closest_cluster = j
        if min_dist < theta:
            C[closest_cluster].add(i)
        else:
            C.append({i})
            m += 1
#es practicamente lo mismo que el BSAS pero sin el limite de numero de clusters por esto se elimina el parametro q
# y la condicion que lo maneja
    return C


Ahora con ayuda de un experimento se detrminan los mejores parámetros para los algoritmos BSAS y MBSAS.

In [192]:
def theta_me_da_m(theta, algoritmo= 'bsas', random_order = True): # esta es una función como f(x) = y 
    if algoritmo == 'bsas':
        clusters = bsas(D, theta, q = len(D), random_order= random_order)
    elif algoritmo == 'mbsas':
        clusters = mbsas(D, theta, random_order= random_order)
    else:
        raise ValueError("Algoritmo no reconocido. Use 'bsas' o 'mbsas'.")
    return len(clusters)

In [193]:
#Ejemplo de uso de la función theta_me_da_m
# con theta = 0.4 y el algoritmo BSAS
m = theta_me_da_m(0.4, algoritmo='bsas')
print(f"Con theta=0.4, el número de clusters es: {m}")

Con theta=0.4, el número de clusters es: 68


In [194]:
#datos para curvas lines 
thetas = np.linspace(0, 1, 200)
ms_mbsas = [theta_me_da_m(theta, algoritmo='mbsas', random_order=True) for theta in thetas]
ms_bsas = [theta_me_da_m(theta, algoritmo='bsas', random_order=True) for theta in thetas]
#Plot de plotly
fig = go.Figure()
# Add BSAS line
fig.add_trace(
    go.Scatter(
        x=thetas,
        y=ms_bsas,
        mode="lines+markers",
        name="BSAS",
        line=dict(width=2),
        marker=dict(size=5),
        hovertemplate="θ=%{x:.3f}<br>m=%{y}<extra></extra>",
        line_color='blue'
    ))
# Add MBSAS line
fig.add_trace(
    go.Scatter(
        x=thetas,
        y=ms_mbsas,
        mode="lines+markers",
        name="MBSAS",
        line=dict(width=2),
        marker=dict(size=5),
        hovertemplate="θ=%{x:.3f}<br>m=%{y}<extra></extra>",
        line_color='orange'
    )
)

# Layout de la figura
fig.update_layout(
    title="Número de clusters vs Umbral de disimilitud (θ)",
    xaxis_title="θ (umbral de disimilitud)",
    yaxis_title="Número de clusters (m)",
    template="plotly_white",
    hovermode="x unified",
    width=950,
    height=520,
    margin=dict(l=60, r=30, t=70, b=50),
)

# Ejes y cuadrícula
fig.update_xaxes(
    showspikes=True,
    spikemode="across",
    spikesnap="cursor",
    showline=True,
    mirror=True,
    gridcolor="rgba(0,0,0,0.08)",
    zeroline=False,
    ),
fig.update_yaxes(
    showline=True,
    mirror=True,
    gridcolor="rgba(0,0,0,0.08)",
    zeroline=False,
)

fig.show()


In [205]:
moda_ms_bsas = pd.Series(ms_bsas).mode().tolist()
print("Moda(s) de m en BSAS:", moda_ms_bsas)
#moda de m en bsas
indices_moda = [i for i, m in enumerate(ms_bsas) if m in moda_ms_bsas]
thetas_por_moda = {m: [] for m in moda_ms_bsas}
for i in indices_moda:
    thetas_por_moda[ms_bsas[i]].append(thetas[i])
m_count = len(indices_moda)
theta_optimo_moda = np.mean(thetas_por_moda[moda_ms_bsas[0]])
print( "Veces que se repite la moda: ", m_count)
print("Theta óptimo (promedio de thetas en la moda):", theta_optimo_moda)

Moda(s) de m en BSAS: [3]
Veces que se repite la moda:  19
Theta óptimo (promedio de thetas en la moda): 0.8635281671515472


In [206]:
#vector de puntos de la cuerda para calcular la distancia a la
cuerda = np.array([[thetas[0], ms_bsas[0]], [thetas[-1], ms_bsas[-1]]])
#dsistancia de cada punto a la cuerda
distancias = np.abs((cuerda[1,1] - cuerda[0,1]) * thetas - (cuerda[1,0] - cuerda[0,0]) * np.array(ms_bsas) + cuerda[1,0]*cuerda[0,1] - cuerda[1,1]*cuerda[0,0]) / np.sqrt((cuerda[1,1] - cuerda[0,1])**2 + (cuerda[1,0] - cuerda[0,0])**2)
#indice del maximo de las distancias
indice_max_distancia = np.argmax(distancias)
#theta correspondiente al maximo de las distancias 
theta_optimo = thetas[indice_max_distancia]
print("Theta óptimo (máxima distancia a la cuerda):", theta_optimo)

Theta óptimo (máxima distancia a la cuerda): 0.3165829145728643


In [207]:
C_BSAS_D = bsas(D, theta_optimo, q = 200, random_order= True)
print(len(C_BSAS_D))
C_BSAS_M = bsas(D, theta_optimo_moda, q = 200, random_order= True)
print(len(C_BSAS_M))

125
3


In [208]:
C_MBSAS_D = mbsas(D, theta_optimo, random_order= True)
print(len(C_MBSAS_D))
C_MBSAS_M = mbsas(D, theta_optimo_moda, random_order= True)
print(len(C_MBSAS_M))

123
3


In [209]:
Y = umap.UMAP(n_components=2, metric='precomputed', random_state=0).fit_transform(D)


c:\Users\narro\AppData\Local\Programs\Python\Python313\Lib\site-packages\umap\umap_.py:1865: UserWarning:

using precomputed metric; inverse_transform will be unavailable

c:\Users\narro\AppData\Local\Programs\Python\Python313\Lib\site-packages\umap\umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.



In [ ]:

n = len(Y)

#cluster asignado a cada punto
labels = np.full(n, -1, dtype=int)
for cid, cluster in enumerate(C_BSAS_M):
    for idx in cluster:
        if 0 <= idx < n:
            labels[idx] = cid

# Convierte a strings para que Plotly los trate como categorías
labels_str = labels.astype(str)
labels_str[labels == -1] = "Unassigned"

# DataFrame para Plotly
df_plot = pd.DataFrame({
    "x": Y[:, 0],
    "y": Y[:, 1],
    "cluster": labels_str
})

# Paleta cualitativa amplia (26 colores). Añadimos gris para "Unassigned".
palette = px.colors.qualitative.Alphabet.copy()
unique_clusters = sorted(df_plot["cluster"].unique(), key=lambda v: (v!="Unassigned", v))

# Mapa discreto cluster->color (si hay más clusters que la paleta, cicla)
color_map = {}
k = len(palette)
count = 0
for cl in unique_clusters:
    color_map[cl] = palette[count % k]
    count += 1

# Scatter interactivo
fig = px.scatter(
    df_plot, x="x", y="y",
    color="cluster",
    color_discrete_map=color_map,
    category_orders={"cluster": unique_clusters},
    title="Embedding con clusters BSAS",
    labels={"x": "UMAP‑1", "y": "UMAP‑2", "cluster": "Cluster ID"},
    hover_data={"cluster": True, "x": ':.3f', "y": ':.3f'}
)

fig.update_traces(marker=dict(size=5, line=dict(width=0)))
fig.update_layout(
    template="plotly_white",
    legend_title_text="Cluster ID",
    legend=dict(itemsizing="constant")
)
fig.show()


In [220]:
#perplexity

perp = 50

tsne = TSNE(
    n_components=2,
    metric='precomputed',
    perplexity=perp,
    random_state=0,
    init='random',
    learning_rate='auto',
    max_iter=1000,
    verbose=1
)
Y_tsne = tsne.fit_transform(D)

# --- Etiquetas de cluster (categóricas) ---
labels = np.full(n, -1, dtype=int)
for cid, cluster in enumerate(C_BSAS_M):
    for idx in cluster:
        if 0 <= idx < n:
            labels[idx] = cid

labels_str = labels.astype(str)
labels_str[labels == -1] = "Unassigned"

# --- DataFrame para plotly ---
df = pd.DataFrame({
    "x": Y_tsne[:, 0],
    "y": Y_tsne[:, 1],
    "cluster": labels_str
})

# Paleta cualitativa amplia + gris para Unassigned
palette = px.colors.qualitative.Alphabet
color_map = {c: palette[i % len(palette)] for i, c in enumerate(sorted(df["cluster"].unique()))}

fig = px.scatter(
    df, x="x", y="y",
    color="cluster",
    color_discrete_map=color_map,
    title=f"t-SNE from precomputed distances (perplexity={perp})",
    labels={"x": "t‑SNE 1", "y": "t‑SNE 2", "cluster": "Cluster ID"},
    hover_data={"x": ':.3f', "y": ':.3f', "cluster": True}
)

fig.update_traces(marker=dict(size=5, line=dict(width=0)))
fig.update_layout(template="plotly_white", legend_title_text="Cluster ID")
fig.show()


[t-SNE] Computing 151 nearest neighbors...
[t-SNE] Indexed 1000 samples in 0.003s...
[t-SNE] Computed neighbors for 1000 samples in 0.023s...
[t-SNE] Computed conditional probabilities for sample 1000 / 1000
[t-SNE] Mean sigma: 0.105552
[t-SNE] KL divergence after 250 iterations with early exaggeration: 58.238495
[t-SNE] KL divergence after 1000 iterations: 0.623610
